# Assignment 9


In [57]:
import os
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json
from pathlib import Path
from itertools import product
import copy
from sklearn.metrics import r2_score
from itertools import product
import matplotlib.pyplot as plt


if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")
    

### Function for spliting train/val/test

In [7]:
def split_csvfiles(datafolder, random_seed, training_prop, validation_prop):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * training_prop)
    val_n = int(len(csv_files) * validation_prop)
    test_n = len(csv_files) - train_n - val_n

    # Split
    if validation_prop == 0:
        train_files = csv_files[:train_n]
        test_files = csv_files[train_n:]

        return train_files, test_files
    
    else:
        train_files = csv_files[:train_n]
        val_files = csv_files[train_n: train_n + val_n]
        test_files = csv_files[train_n + val_n:]

        return train_files, val_files, test_files

### Load data

In [13]:
def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)

        # Get rid of trailing whitespace
        df.columns = df.columns.str.strip()           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined

### Function for splitting input (x, y) and target (z)

In [32]:
def input_target_split(dataframe):
    input_col_names = []
    target_col_names = []

    joints = ["head", "left_shoulder", "left_elbow", "right_shoulder", "right_elbow", "left_hand", "right_hand",
              "left_hip", "right_hip", "left_knee", "right_knee", "left_foot", "right_foot"]
    
    for joint in joints:
        input_col_names += [f"{joint}_x", f"{joint}_y"]
        target_col_names += [f"{joint}_z"]


    input_data = dataframe[input_col_names].copy()
    target_data = dataframe[target_col_names].copy()
    
    return input_data, target_data

### Load data

In [33]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

# Alt 1: 80/10/10 split
train_files, val_files, test_files = split_csvfiles(datafolder, random_seed, 0.8, 0.1)
val_data = load(val_files, datafolder)

# Alt 2: 90/10 since we don't need validation data if we run k-fold cv
# train_files, test_files = split_csvfiles(datafolder, random_seed, 0.9, 0)

train_data = load(train_files, datafolder)
test_data = load(test_files, datafolder)

print(train_data.columns.tolist())

x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)



x_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
x_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

['FrameNo', 'head_x', 'head_y', 'head_z', 'left_shoulder_x', 'left_shoulder_y', 'left_shoulder_z', 'left_elbow_x', 'left_elbow_y', 'left_elbow_z', 'right_shoulder_x', 'right_shoulder_y', 'right_shoulder_z', 'right_elbow_x', 'right_elbow_y', 'right_elbow_z', 'left_hand_x', 'left_hand_y', 'left_hand_z', 'right_hand_x', 'right_hand_y', 'right_hand_z', 'left_hip_x', 'left_hip_y', 'left_hip_z', 'right_hip_x', 'right_hip_y', 'right_hip_z', 'left_knee_x', 'left_knee_y', 'left_knee_z', 'right_knee_x', 'right_knee_y', 'right_knee_z', 'left_foot_x', 'left_foot_y', 'left_foot_z', 'right_foot_x', 'right_foot_y', 'right_foot_z']


### Model

### Saving model

In [53]:
# Base paths
base_model_dir = "../assignment9_models"
candidates_dir = os.path.join(base_model_dir, "candidates")
champion_dir = os.path.join(base_model_dir, "champion")
metadata_dir = os.path.join(base_model_dir, "metadata")

# Create folders if they don't exist
os.makedirs(candidates_dir, exist_ok=True)
os.makedirs(champion_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)


# =========================
# SAVE CANDIDATE MODEL
# =========================
def save_candidate_model(model, model_name):
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"Saved candidate model: {path}")
    return path


# =========================
# LOAD CHAMPION INFO
# =========================
def load_champion_info():
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None  


# =========================
# SAVE NEW CHAMPION
# =========================
def save_champion_model(model, model_name, mse, mae, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mse": float(mse),
        "mae": float(mae),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# =========================
# UPDATE CHAMPION LOGIC
# =========================
def update_champion(model, model_name, mse, mae, hyperparameters):
    current = load_champion_info()

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    elif mse < current["mse"]:
        print(f"New model is better (MSE {mse} < {current['mse']})")
        save_champion_model(model, model_name, mse, mae, hyperparameters)

    else:
        print(f"Model NOT better (MSE {mse} ≥ {current['mse']})")

### Functions for ML

In [50]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation="relu", dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)

        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        act = activations[activation]
        prev_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(act)

            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 13))  # 13 joints z output
        
        self.network = nn.Sequential(*layers)

        self.network.apply(init_weights)
    
    def forward(self, x):
        return self.network(x)


def build_model(config):
    return ZPredictor(
        hidden_layers=config["hidden_layers"],
        activation=config["activation"],
        dropout=config["dropout"],
    ).to(device)

def compute_metrics(y_true, y_pred):
    mse = torch.mean((y_pred - y_true) ** 2).item()
    mae = torch.mean(torch.abs(y_pred - y_true)).item()
    bias = torch.mean(y_pred - y_true).item()

    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()
    r2 = r2_score(y_true_np, y_pred_np)

    return {"mse": mse, "mae": mae, "r2": r2, "bias": bias}

def evaluate_model(model, x_data, y_data, loss_fn):
    model.eval()
    with torch.no_grad():
        predictions = model(x_data)
        loss = loss_fn(predictions, y_data).item()
        metrics = compute_metrics(y_data, predictions)
    return loss, metrics, predictions


def train_one_model(model, config, x_train, y_train, x_val, y_val, verbose=False):
    optimizer_name = config["optimizer"]
    lr = config["learning_rate"]

    if optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    loss_fn = nn.MSELoss()
    epochs = config["epochs"]
    patience = config["patience"]

    best_val_loss = float("inf")
    best_state = None
    history = []
    epochs_no_improve = 0

    for epoch in range(epochs):
        # Train
        model.train()
        optimizer.zero_grad()
        train_predictions = model(x_train)
        train_loss = loss_fn(train_predictions, y_train)
        train_loss.backward()
        optimizer.step()

        # Evaluate
        val_loss, val_metrics, _ = evaluate_model(model, x_val, y_val, loss_fn)
        train_metrics = compute_metrics(y_train, train_predictions)

        row = {
            "epoch": epoch + 1,
            "train_loss": float(train_loss.item()),
            "val_loss": float(val_loss),
            "train_mae": train_metrics["mae"],
            "val_mae": val_metrics["mae"],
            "train_r2": train_metrics["r2"],
            "val_r2": val_metrics["r2"],
        }
        history.append(row)



        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(
                f"Epoch {epoch+1:03d} | "
                f"train_loss={train_loss.item():.6f} | "
                f"val_loss={val_loss:.6f} | "
                f"val_mae={val_metrics['mae']:.6f}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_loss, final_val_metrics, _ = evaluate_model(model, x_val, y_val, loss_fn)
    return {
        "model": model,
        "best_state": best_state,
        "history": history,
        "val_loss": final_val_loss,
        "val_metrics": final_val_metrics,
    }

## Hyperparameter search

In [77]:
PARAM_GRID = {
    "hidden_layers": [
        [256, 128],
        [256, 128, 64],
        [512, 256, 128],
    ],
    "learning_rate": [1e-3, 5e-4],
    "dropout": [0.0, 0.05, 0.1],
    "activation": ["relu", "leaky_relu"],
    "optimizer": ["adam"],
    "epochs": [100],
    "patience": [5, 10],
}

search_space = list(product(
    PARAM_GRID["hidden_layers"],
    PARAM_GRID["learning_rate"],
    PARAM_GRID["dropout"],
    PARAM_GRID["activation"],
    PARAM_GRID["optimizer"],
    PARAM_GRID["epochs"],
    PARAM_GRID["patience"],
))


print(f"Number of runs: {len(search_space)}")

Number of runs: 72


In [78]:
best_result = None
all_results = []

for run_idx, values in enumerate(search_space, start=1):
    config = {
        "hidden_layers": values[0],
        "learning_rate": values[1],
        "dropout": values[2],
        "activation": values[3],
        "optimizer": values[4],
        "epochs": values[5],
        "patience": values[6],
    }

    run_name = (
        f"run_{run_idx}_"
        f"layers_{'-'.join(map(str, config['hidden_layers']))}_"
        f"lr_{config['learning_rate']}_"
        f"dropout_{config['dropout']}_"
        f"act_{config['activation']}"
    )

    active_run = None


    model = build_model(config)
    result = train_one_model(model, config, x_train, y_train, x_val, y_val, verbose=False)

    val_loss = result["val_loss"]
    val_metrics = result["val_metrics"]

    # save_candidate_model(result["model"], run_name)

    summary = {
        "run_name": run_name,
        **config,
        "val_loss": val_loss,
        "val_mse": val_metrics["mse"],
        "val_mae": val_metrics["mae"],
        "val_r2": val_metrics["r2"],
        "val_bias": val_metrics["bias"],
        "model_state_dict": copy.deepcopy(result["model"].state_dict()),
    }

  
    if best_result is None or val_metrics["mae"] < best_result["val_mae"]:
        best_result = summary


        update_champion(
            model=result["model"],
            model_name="assignment9_best_model",
            mse=val_metrics["mse"],
            mae=val_metrics["mae"],
            hyperparameters=config,
        )

    

    print(
        f"[{run_idx:02d}/{len(search_space)}] {run_name} -> "
        f"val_mae={val_metrics['mae']:.6f}, val_loss={val_loss:.6f}"
    )

print("\nBest validation run:")
print({k: v for k, v in best_result.items() if k != "model_state_dict"})

Model NOT better (MSE 0.05316302925348282 ≥ 0.004888102412223816)
[01/72] run_1_layers_256-128_lr_0.001_dropout_0.0_act_relu -> val_mae=0.184511, val_loss=0.053163
Model NOT better (MSE 0.005779511760920286 ≥ 0.004888102412223816)
[02/72] run_2_layers_256-128_lr_0.001_dropout_0.0_act_relu -> val_mae=0.053682, val_loss=0.005780
[03/72] run_3_layers_256-128_lr_0.001_dropout_0.0_act_leaky_relu -> val_mae=0.148416, val_loss=0.034180
[04/72] run_4_layers_256-128_lr_0.001_dropout_0.0_act_leaky_relu -> val_mae=0.058039, val_loss=0.006759
[05/72] run_5_layers_256-128_lr_0.001_dropout_0.05_act_relu -> val_mae=0.159386, val_loss=0.041495
[06/72] run_6_layers_256-128_lr_0.001_dropout_0.05_act_relu -> val_mae=0.070849, val_loss=0.008609
[07/72] run_7_layers_256-128_lr_0.001_dropout_0.05_act_leaky_relu -> val_mae=0.195695, val_loss=0.066087
[08/72] run_8_layers_256-128_lr_0.001_dropout_0.05_act_leaky_relu -> val_mae=0.055066, val_loss=0.005701
[09/72] run_9_layers_256-128_lr_0.001_dropout_0.1_act_r

## Final model on test set

In [79]:
best_config = {
    "hidden_layers": best_result["hidden_layers"],
    "learning_rate": best_result["learning_rate"],
    "dropout": best_result["dropout"],
    "activation": best_result["activation"],
    "optimizer": best_result["optimizer"],
    "epochs": best_result["epochs"],
    "patience": best_result["patience"],
}

best_model = build_model(best_config)
best_model.load_state_dict(best_result["model_state_dict"])

loss_fn = nn.MSELoss()

print("Best config:")
print(best_config)


# Train
train_loss, train_metrics, train_predictions = evaluate_model(best_model, x_train, y_train, loss_fn)

print("\nTrain metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.6f}")
print(f"test_loss: {train_loss:.6f}")


# Validation
val_loss, val_metrics, val_predictions = evaluate_model(best_model, x_val, y_val, loss_fn)

print("\nValidation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.6f}")
print(f"test_loss: {val_loss:.6f}")


Best config:
{'hidden_layers': [512, 256, 128], 'learning_rate': 0.001, 'dropout': 0.0, 'activation': 'relu', 'optimizer': 'adam', 'epochs': 100, 'patience': 10}

Train metrics:
mse: 0.003878
mae: 0.046356
r2: 0.605922
bias: -0.000344
test_loss: 0.003878

Validation metrics:
mse: 0.004936
mae: 0.049495
r2: 0.354598
bias: -0.010583
test_loss: 0.004936


In [80]:
test_loss, test_metrics, test_predictions = evaluate_model(best_model, x_test, y_test, loss_fn)

print("Best config:")
print(best_config)
print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.6f}")
print(f"test_loss: {test_loss:.6f}")

Best config:
{'hidden_layers': [512, 256, 128], 'learning_rate': 0.001, 'dropout': 0.0, 'activation': 'relu', 'optimizer': 'adam', 'epochs': 100, 'patience': 10}

Test metrics:
mse: 0.004409
mae: 0.049451
r2: 0.453276
bias: 0.010844
test_loss: 0.004409
